# pyRockImageLab — Example: core box photo to high-resolution mosaic

A small, anonymized example (`example_data/ExampleCore/`) that reproduces the 2563–2566 ft high-resolution (HR) core mosaic.

- `ExampleCore_00_box.jpg`: overview photo of the core box (three 3-ft segments)
- `ExampleCore_01..06_detail.jpg`: close-up photos of segment 1 (2563–2566 ft)
- `_picks.json`: saved corner picks, so the notebook runs without clicking (made with the app, `pyrockimagelab-app`)

Workflow: crop and ortho-warp each detail photo → stitch them (SIFT + RANSAC) → ortho-warp the mosaic → save PNG.
Outputs go to `outputs/` as PNG only.

In [1]:
# %pip install --quiet pyrockimagelab pillow-heif

In [2]:
from pathlib import Path
import matplotlib
import pyrockimagelab as pril
from pyrockimagelab import (list_source_images, load_any_image, load_picks,
                            ortho_warp_quad, stitch_chain, release_working_memory)
from pyrockimagelab.image_processing.warp import _apply_hdr_enhancement, clamp_quad_to_image, trim_empty_rows
import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.patches

FOLDER = Path("example_data/ExampleCore/2563-2572")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)      # PNG outputs (not tracked by git)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)       # small README figure
HR_SAVE_SCALE = 0.5   # 1.0 = full resolution (~50 MB PNG); 0.5 keeps it ~13 MB
picks = load_picks(FOLDER)
sources = list_source_images(FOLDER)
print(f"pyrockimagelab {pril.__version__}")
for i, p in enumerate(sources):
    print(f"[{i}] {p.name}")

pyrockimagelab 0.0.3
[0] ExampleCore_00_box.jpg
[1] ExampleCore_01_detail.jpg
[2] ExampleCore_02_detail.jpg
[3] ExampleCore_03_detail.jpg
[4] ExampleCore_04_detail.jpg
[5] ExampleCore_05_detail.jpg
[6] ExampleCore_06_detail.jpg


## Step 1 — Box photo and section 1

The saved corners mark core section 1 on the box photo (TL/TR/BR/BL = top-left, top-right, bottom-right, bottom-left). To mark corners yourself, use the app (last cell).

In [3]:
box = load_any_image(FOLDER / picks["box_photo"])
quad = picks["box_quads"][f'{picks["box_photo"]}::seg1']

lr = _apply_hdr_enhancement(ortho_warp_quad(box, quad))
Image.fromarray(lr).save(OUT / "ExampleCore_2563-2566.png", optimize=True)
print("LR segment:", lr.shape[1], "x", lr.shape[0], "px")

# Figure A: box photo with picked corners -> cropped, orthorectified segment (true aspect ratios)
H_IN = 6.5                                    # both panels drawn at this height (inches)
bw = H_IN * box.shape[1] / box.shape[0]
sw = H_IN * lr.shape[1] / lr.shape[0]
L, GAP, R, B, T = 0.4, 2.0, 0.9, 0.4, 0.8     # margins / gap (inches)
FW, FH = L + bw + GAP + sw + R, B + H_IN + T
fig = plt.figure(figsize=(FW, FH), facecolor="white")
a0 = fig.add_axes([L / FW, B / FH, bw / FW, H_IN / FH])
a0.imshow(box[::4, ::4], extent=(0, box.shape[1], box.shape[0], 0))
q = np.array(quad + [quad[0]])
a0.plot(q[:, 0], q[:, 1], color="yellow", lw=1.5)
for (x, y), lab in zip(quad, ("TL", "TR", "BR", "BL")):
    a0.plot(x, y, "o", ms=5, mfc="yellow", mec="k")
    a0.annotate(lab, (x, y), xytext=(6, -6 if lab[0] == "T" else 8), textcoords="offset points",
                color="yellow", fontsize=8, weight="bold")
a0.set_title("Box photo with segment corners picked", fontsize=10)
a0.axis("off")
a1 = fig.add_axes([(L + bw + GAP) / FW, B / FH, sw / FW, H_IN / FH])
a1.imshow(lr[::2, ::2], extent=(0, lr.shape[1], lr.shape[0], 0))
a1.set_xticks([]); a1.set_yticks([0, lr.shape[0]]); a1.set_yticklabels(["2563 ft", "2566 ft"], fontsize=8)
a1.yaxis.tick_right()
for sp in a1.spines.values(): sp.set_visible(False)
fig.text((L + bw + GAP + sw / 2) / FW, (B + H_IN + 0.15) / FH, "Cropped & orthorectified\nsegment (2563–2566 ft)",
         ha="center", va="bottom", fontsize=10)
y = (B + H_IN / 2) / FH
fig.add_artist(matplotlib.patches.FancyArrowPatch(((L + bw + 0.2) / FW, y), ((L + bw + GAP - 0.2) / FW, y),
               transform=fig.transFigure, arrowstyle="-|>", mutation_scale=18, lw=1.5, color="0.2"))
fig.text((L + bw + GAP / 2) / FW, y + 0.25 / FH, "crop +\northorectify", ha="center", va="bottom", fontsize=9)
fig.savefig(FIG / "ExampleCore_2563-2566_box_crop.png", dpi=110, facecolor="white")
plt.close(fig)

LR segment: 305 x 3610 px


## Step 2 — Trim each close-up photo with its saved corners

In [4]:
names = [sources[i].name for i in picks["hr_assignments"]["seg1"]]
crops = []
for n in names:
    img = load_any_image(FOLDER / n)
    q = picks["hr_photo_quads"][n]
    crops.append(_apply_hdr_enhancement(ortho_warp_quad(img, q)))
    print(f"{n}: {crops[-1].shape[1]} x {crops[-1].shape[0]} px")

ExampleCore_01_detail.jpg: 1627 x 3777 px


ExampleCore_02_detail.jpg: 1588 x 4011 px


ExampleCore_03_detail.jpg: 1582 x 4020 px


ExampleCore_04_detail.jpg: 1603 x 4017 px


ExampleCore_05_detail.jpg: 1633 x 4017 px


ExampleCore_06_detail.jpg: 1708 x 3468 px


## Step 3 — Join (stitch), straighten and save the high-resolution image

The saved corners of the joined image are clipped to the image and empty rows are trimmed, so the result has no black margin and spans the same core interval as the box-photo section.

In [5]:
mosaic = stitch_chain(1, crops)
quad = clamp_quad_to_image(picks["hr_mosaic_quad_seg1"], mosaic)
hr = trim_empty_rows(ortho_warp_quad(mosaic, quad))
print("HR mosaic:", hr.shape[1], "x", hr.shape[0], "px")
h, w = hr.shape[:2]
Image.fromarray(hr).resize((round(w * HR_SAVE_SCALE), round(h * HR_SAVE_SCALE)), Image.LANCZOS) \
    .save(OUT / "ExampleCore_2563-2566_hr.png", optimize=True)

# Figure B: six close-up crops (true aspect, uniform scale, 2 x 3 grid) -> stitched HR mosaic (true aspect)
n = len(crops)
CH = 3.3                                      # crop height (inches) in the grid
cw = [CH * c.shape[1] / c.shape[0] for c in crops]
COLW, PAD = max(cw), 0.15
L, B, T, GAP, R = 0.4, 0.4, 0.9, 1.6, 0.9
GW = 3 * COLW + 2 * PAD
GH = 2 * CH + 0.35                            # 2 rows + room for numbers
MH = GH                                       # mosaic drawn at grid height
mw = MH * hr.shape[1] / hr.shape[0]
FW, FH = L + GW + GAP + mw + R, B + GH + T
fig = plt.figure(figsize=(FW, FH), facecolor="white")
for k, c in enumerate(crops):
    r, col = divmod(k, 3)
    x0 = L + col * (COLW + PAD) + (COLW - cw[k]) / 2
    y0 = B + (1 - r) * (CH + 0.35)
    a = fig.add_axes([x0 / FW, y0 / FH, cw[k] / FW, CH / FH])
    a.imshow(c[::3, ::3]); a.axis("off")
    a.text(0.5, 1.02, str(k + 1), transform=a.transAxes, ha="center", va="bottom", fontsize=10, weight="bold")
fig.text((L + GW / 2) / FW, (B + GH + 0.3) / FH, "Close-up crops (before stitching, in order 1–6)",
         ha="center", va="bottom", fontsize=11)
am = fig.add_axes([(L + GW + GAP) / FW, B / FH, mw / FW, MH / FH])
am.imshow(hr[::4, ::4], extent=(0, hr.shape[1], hr.shape[0], 0))
am.set_xticks([]); am.set_yticks([0, hr.shape[0]]); am.set_yticklabels(["2563 ft", "2566 ft"], fontsize=8)
am.yaxis.tick_right()
for sp in am.spines.values(): sp.set_visible(False)
fig.text((L + GW + GAP + mw / 2) / FW, (B + GH + 0.3) / FH, "Stitched HR mosaic\n(2563–2566 ft)",
         ha="center", va="bottom", fontsize=11)
y = (B + GH / 2) / FH
fig.add_artist(matplotlib.patches.FancyArrowPatch(((L + GW + 0.2) / FW, y), ((L + GW + GAP - 0.2) / FW, y),
               transform=fig.transFigure, arrowstyle="-|>", mutation_scale=18, lw=1.5, color="0.2"))
fig.text((L + GW + GAP / 2) / FW, y + 0.2 / FH, "stitch", ha="center", va="bottom", fontsize=10)
fig.savefig(FIG / "ExampleCore_2563-2566_hr.png", dpi=110, facecolor="white")
plt.close(fig)
del mosaic, crops
release_working_memory()

  pair 0->1:


    25001+25000 desc, 25001 raw, 294 good
      RANSAC: 180/294 inliers, persp=1.20e-05
    -> accepted (180 inliers)
  pair 1->2:


    25000+25001 desc, 25000 raw, 78 good
      RANSAC: 39/78 inliers, persp=1.11e-05
    -> accepted (39 inliers)
  pair 2->3:


    25001+25000 desc, 25001 raw, 441 good
      RANSAC: 418/441 inliers, persp=9.02e-06
    -> accepted (418 inliers)
  pair 3->4:


    25000+25001 desc, 25000 raw, 80 good
      RANSAC: 47/80 inliers, persp=1.05e-05
    -> accepted (47 inliers)
  pair 4->5:


    25001+25001 desc, 25001 raw, 54 good
      RANSAC: 22/54 inliers, persp=2.17e-05
    -> accepted (22 inliers)
  Blending 6 images...


HR mosaic: 1571 x 15935 px


  Memory cleared.


### Mark corners yourself (browser app)

`pyrockimagelab-app` opens in your browser. Pick **Crop core-box photo** or **Stitch close-up photos**, type the project folder (here `example_data`), and click `ExampleCore / 2563-2572`. Everything you mark is saved in the same `_picks.json`, so the cells above (or `pril.process_folder`) replay it.

From a terminal: `pyrockimagelab-app example_data`

In [ ]:
# import subprocess, sys
# app = subprocess.Popen([sys.executable, "-m", "pyrockimagelab.webapp", "example_data"])   # app.terminate() to stop